# ParkFly P3: real FlyWire connectome
No FlyWire token needed. Run cells in order. Six sanity checks must pass. Lesion cuts Giant Fibre input columns. Coarse results are exported before the full grid; raw runs checkpoint every ten completions.
The connectome is a fixed spiking filter, not a trained parking model. Outputs are rates and their SD across fixed seeds.

In [ ]:
from pathlib import Path
import os, sys
if not Path('/content/parkfly/.git').exists():
    !git clone -q https://github.com/momacaes9/parkfly.git /content/parkfly
%cd /content/parkfly
%pip -q install numpy pandas scipy pyarrow matplotlib
sys.path.insert(0, '/content/parkfly/pipeline')
import fly_surface as fs
import pandas as pd
Path(fs.OUT).mkdir(parents=True, exist_ok=True)
sanity = fs.sanity()
display(pd.DataFrame(sanity['checks']))
assert sanity['all_pass'], 'STOP: fix wiring before the surface.'


In [ ]:
import shutil
from google.colab import files
fs.GRID = {'sugar':[0,50,100,150,200], 'bitter':[0,50,100,200], 'loom':[0,50,100,150]}
fs.surface(workers=2, seeds=3, coarse=True)
coarse = Path(fs.OUT) / 'fly_surface.coarse.json'
shutil.copyfile(Path(fs.OUT) / 'fly_surface.json', coarse)
files.download(str(coarse))
fs.surface(workers=2, seeds=3, coarse=False)
files.download(str(Path(fs.OUT) / 'fly_surface.json'))
files.download(str(Path(fs.OUT) / 'fly_sanity.json'))


In [ ]:
# Optional P3 step 3: recorded spike rasters for three canonical scenarios.
fs.raster()
files.download(str(Path(fs.OUT) / 'fly_raster.json'))


## Optional Giant Fibre skeleton
The uploaded NC State starter uses FAFBseg 3.2.3, Navis 1.12.0, Plotly and crackle-codec for this visualization. The full NumPy/SciPy simulation above needs no token. This optional cell remains off until you explicitly enable it. Never save a token in notebook source or output.

In [ ]:
RUN_SKELETON_VIEW = False
if RUN_SKELETON_VIEW:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "fafbseg==3.2.3", "navis==1.12.0", "plotly", "crackle-codec"])
    from getpass import getpass
    from fafbseg import flywire
    import navis
    token = getpass("Private FlyWire token (only for the optional skeleton): ").strip()
    if not token:
        print("No token entered; skipping visualization. Brain simulation remains available.")
    else:
        flywire.set_chunkedgraph_secret(token, overwrite=True)
        flywire.set_default_dataset("public")
        del token
        annotations = flywire.search_annotations(flywire.NeuronCriteria(type="DNp01"))
        navis.plot3d(flywire.get_skeletons(annotations.root_id.iloc[0]))
